In [ ]:
pip install tensorflow opencv-python numpy pandas matplotlib seaborn scikit-learn streamlit pillow kagglehub

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input/datasets/anourhamuze/neurovisiondata"):
    print(f"\nDossier : {root}")
    print(f"Sous-dossiers : {dirs}")
    print(f"Nombre de fichiers : {len(files)}")

In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("anourhamuze/neurovisiondata")
print("Path to dataset files:", path)

In [ ]:
from pathlib import Path
# data_path = Path("/kaggle/input/datasets/anourhamuze/neurovisiondata/Data")
data_path = Path(os.path.join(path, 'Data'))
extensions = {".jpg", ".jpeg", ".png", ".bmp"}
# classes = np.array(sorted([item.name for item in data_path.glob("*")]))
# print(classes)
classes = sorted(dossier.name for dossier in data_path.iterdir()if dossier.is_dir())
print(classes)
image_diff_ex=0
resultats = []
etiquette = []
images = []
for classe in classes:
    dossier = data_path/classe
    for f in dossier.rglob("*"):
      if f.is_file():
        if f.suffix.lower() in extensions:
            images.append(f)
            etiquette.append(classe)
        else:
            image_diff_ex += 1
    resultats.append({"Classe": classe, "Nombre d'images": len(images)})
print(resultats)
# print(image_diff_ex)

In [ ]:
images_invalides = []
for img in images:
    try:
        image = cv2.imread(str(img))
        if image is None:
            images_invalides.append(str(img))
            continue
    except Exception as e:
        images_invalides.append(str(img))
        print(f"Erreur pour {f} : {e}")
print(images_invalides)

In [ ]:
# import random
# combined = list(zip(images, etiquette))
# random.seed(42)
# random.shuffle(combined)
# images, labels = zip(*donnees)
# images = list(images)
# labels = list(labels)
# print(combined)

In [ ]:

# images_redimensionnees = []
# etiquettes_valides = []

# for chemin, label in zip(images, etiquette):
#     image = cv2.imread(str(chemin))

#     if image is not None:
#         image_resize = cv2.resize(image, (224, 224))
#         images_redimensionnees.append(image_resize)
#         etiquettes_valides.append(label)

# images = np.array(images_redimensionnees)
# etiquette = np.array(etiquettes_valides)



In [ ]:
# images_redimensionnees = []
# for img in images:
#     image = cv2.imread(str(img))
#     image_resize = cv2.resize(image, (224, 224))
#     images_redimensionnees.append(image_resize)
# images = np.array(images_redimensionnees)
# etiquette = np.array(etiquette)
# print(etiquette)


In [ ]:
def charger_images(images):
    images_charge = []
    etiquettes = []
    for img, label in zip(images, etiquette):
        image = cv2.imread(str(img))
        image = cv2.resize(image, (224, 224))
        images_charge.append(image)
        etiquettes.append(label)
    return images_charge, etiquettes

In [ ]:
images, etiquette = charger_images(images)

import random

combined = list(zip(images, etiquette))
random.seed(42)
random.shuffle(combined)
images, labels = zip(*combined)
images = list(images)
labels = list(labels)


In [ ]:
import numpy as np
X1 = np.array(images)
y = np.array(labels)
# print(labels)
elements, counts = np.unique(y, return_counts=True)
for element, count in zip(elements, counts):
    print(f"{element}: {count}")

In [ ]:
sns.countplot(x=y, order=classes)

plt.title("Repartition des images par classe")
plt.xlabel("Classe")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=15)
plt.show()

In [ ]:

classes = sorted(np.unique(y))
for class_name in classes:
    # print(np.where(y == class_name)[0])
    index_list_imges = np.where(y == class_name)[0]
    print(index_list_imges[0])
    plt.imshow(X1[index_list_imges[0]])
    plt.title(class_name)
    plt.axis("off")
    plt.show()

In [ ]:
total = len(y)
print(total)

for classe, count in zip(elements, counts):
    percentage = count/total*100
    print(f"{classe} has {count} images cad : {percentage.round(2)}%")
# df_classes = pd.DataFrame({
#     "Classe": classes,
#     "Nombre d'images": counts,
#     "Pourcentage": counts / len(y) * 100
# })
# df_classes

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
y_encoded = le.fit_transform(y)

num_class = len(le.classes_)
# print(num_class)
y_categorical = to_categorical(
    y_encoded,
    num_classes = num_class
)
for i, class_name in enumerate(le.classes_):
    print(class_name, "=", i)
print(y_categorical.shape)
print(y_categorical)


In [ ]:
image_before = X1[0].copy()
image_after = image_before.astype(np.float32) / 255.0
X = X1.astype(np.float32) / 255.0
print(image_before)
print("helllo")
print(image_after)
plt.imshow(image_before)
plt.title("image before")
plt.axis("off")
plt.show()

plt.imshow(image_after)
plt.title("image after")
plt.axis("off")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_categorical,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test :", y_test.shape)


In [ ]:
from tensorflow.keras import layers

augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.04), 
    layers.RandomZoom(0.05), 
    layers.RandomTranslation(height_factor = 0.03, width_factor=0.03)
], name="augmentation")

augmented_sample = augmentation(X_train[0], training=True)

plt.imshow(X_train[0])
plt.title("Image originale")
plt.axis("off")
plt.show()

plt.imshow(augmented_sample)
plt.title("Image augmentee")
plt.axis("off")
plt.show()

In [ ]:
batch_size = 30
Autotun = tf.data.AUTOTUNE
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(buffer_size=len(X_train))
train_ds = train_ds.map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=Autotun)
train_ds = train_ds.batch(batch_size).prefetch(Autotun)

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import time

from tensorflow.keras import layers, models
from tensorflow.keras.utils import plot_model

In [ ]:
cnn_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(32, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(),
    
    layers.Conv2D(128, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(),
    
    layers.Conv2D(192, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(),
    
    layers.GlobalAveragePooling2D(),
    
    layers.Flatten(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.4),

    layers.Dense(4, activation="softmax")
], name="cnn_model")
cnn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
cnn_model.summary()

In [ ]:
pip install pydot

In [ ]:
from tensorflow.keras.utils import plot_model
from IPython.display import Image, display
from pathlib import Path

output_path = Path.cwd() / "neurovision_cnn.png"

plot_model(
    cnn_model,
    to_file=str(output_path),
    show_shapes=True,
    show_layer_names=True,
    dpi=100
)

if output_path.exists():
    display(Image(filename=str(output_path)))
else:
    print("Le fichier n'a pas ete genere.")

In [ ]:
epochs = 15
start_time = time.time()

history = cnn_model.fit(
    train_ds,
    epochs = epochs,
)
end_time = time.time()

training_duration = end_time - start_time
print(f"Duree totale : {training_duration / 60:.2f} minutes")

In [ ]:
plt.plot(history.history["accuracy"], label="Precision entrainement")
plt.plot(history.history["loss"], label="Loss entrainement")
plt.title("Evolution pendant l'entrainement")
plt.xlabel("Epoque")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
)
print("Entrainement :", X_train_final.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)


# def prepare_image(image, label):
#     image = tf.cast(image, tf.float32) / 255.0
#     return image, label


train_ds = tf.data.Dataset.from_tensor_slices((X_train_final, y_train_final))
train_ds = train_ds.shuffle(len(X_train_final))
train_ds = train_ds.map(lambda image, label: (augmentation(image, training=True),label),num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
# test_ds = test_ds.map(prepare_image,num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


In [ ]:

from pathlib import Path
from tensorflow.keras.callbacks import ModelCheckpoint, BackupAndRestore, EarlyStopping, ReduceLROnPlateau

Path("models").mkdir(parents=True, exist_ok=True)
Path("training_backupp").mkdir(parents=True, exist_ok=True)

checkpoint = ModelCheckpoint(
    filepath="models/models/best_neurovision_v2.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

backup = BackupAndRestore(
    backup_dir="training_backupp",
    save_freq="epoch"
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    mode="min",
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    mode="min",
    verbose=1
)

callbacks = [
    checkpoint,
    backup,
    early_stopping,
    reduce_lr
]

start_time = time.time()
history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)
end_time = time.time()
duration = end_time - start_time
print(f"Duree : {duration / 60:.2f} minutes")



In [ ]:
plt.plot(
    history.history["accuracy"],
    label="Entrainement"
)
plt.plot(
    history.history["val_accuracy"],
    label="Validation"
)
plt.title("evolution de l'accuracy")
plt.xlabel("epoque")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.plot(
    history.history["loss"],
    label="Entrainement"
)
plt.plot(
    history.history["val_loss"],
    label="Validation"
)
plt.title("evolution de la loss")
plt.xlabel("epoque")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
cnn_best_model = tf.keras.models.load_model("models/models/best_neurovision_v2.keras")
test_loss, test_accuracy = cnn_best_model.evaluate(test_ds)
print(f"Test loss : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy * 100:.2f}%")

In [ ]:
# ===========================================================part2==================


In [ ]:
import os

OUTPUT_DIR = os.path.join(os.getcwd(), "Output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output directory:", OUTPUT_DIR)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model

IMG_SHAPE = (224,224,3)
inputs = layers.Input(shape = IMG_SHAPE)
x = layers.Rescaling(scale = 2.0, offset = -1.0)(inputs)
base_model = MobileNetV2( weights ='imagenet', include_top = False, input_shape =IMG_SHAPE)
base_model.trainable = False
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(4, activation='softmax')(x)
transfer_lerning_model = Model(inputs = inputs, outputs = outputs, name="transfer_lerning_model")
transfer_lerning_model.summary()

In [ ]:
transfer_lerning_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate =0.001 ), 
    loss= 'categorical_crossentropy', 
    metrics = ['accuracy']
    )

In [ ]:
print("Parametres totales :", transfer_lerning_model.count_params())
print("Parametres entrainables :", sum(tf.keras.backend.count_params(w) for w in transfer_lerning_model.trainable_weights))
print("Parametres non entrainables :",sum(tf.keras.backend.count_params(w) for w in transfer_lerning_model.non_trainable_weights))

In [ ]:
start_time = time.time()
history = transfer_lerning_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
)
end_time = time.time()
duration = end_time - start_time
print(f"duree d'entrainement : {duration / 60:.2f} minutes")

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

loss = history.history["loss"]
val_loss = history.history["val_loss"]


plt.plot(acc, label="train Accuracy")
plt.plot(val_acc, label="validation Accuracy")
plt.title("evolution de l'accuracy")
plt.xlabel("epoque")
plt.ylabel("accuracy")
plt.legend()
plt.show()

plt.plot(loss, label="train Loss")
plt.plot(val_loss, label="validation Loss")
plt.title("evolution de la loss")
plt.xlabel("epoque")
plt.ylabel("loss")
plt.legend()

plt.show()

In [ ]:
transfer_lerning_model.save(OUTPUT_DIR, "neurovision_mobilenetv2.keras")
print("modele charge localement dans :", OUTPUT_DIR)

In [ ]:
from tensorflow.keras.models import load_model
fine_tuning_model = load_model(f"{OUTPUT_DIR}/neurovision_mobilenetv2.keras")

In [ ]:
before_loss, before_accuracy = fine_tuning_model.evaluate(val_ds)
print(f"Avant fine-tuning val accuracy : {before_accuracy:.2f}")
print(f"Avant fine-tuning val loss     : {before_loss:.4f}")

In [ ]:
fine_tuning_model.trainable = True

for layer in fine_tuning_model.layers:
    layer.trainable = False
for layer in fine_tuning_model.layers[-35:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True
# fine_tuning_model = tf.keras.Model(
#     inputs=inputs,
#     outputs=outputs,
#     name="MobileNetV2_Fine_Tuning_Last35"
# )
print("couches totales:", len(fine_tuning_model.layers))
print("couches entrainables :", sum(layer.trainable for layer in fine_tuning_model.layers))
print("couches gelees :", sum(not layer.trainable for layer in fine_tuning_model.layers))

In [ ]:
fine_tuning_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
    
)
fine_tuning_model.summary()

In [ ]:

Path("models_v2").mkdir(parents=True, exist_ok=True)
Path("training_backupp_v2").mkdir(parents=True, exist_ok=True)

checkpoint = ModelCheckpoint(
    filepath="models/models_v2/best_neurovision.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

backup = BackupAndRestore(
    backup_dir="training_backupp_v2",
    save_freq="epoch"
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    mode="min",
    verbose=1
)

callbacks = [
    checkpoint,
    backup,
    early_stopping
]

start_time = time.time()
history = fine_tuning_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)
end_time = time.time()
duration = end_time - start_time
print(f"Duree : {duration / 60:.2f} minutes")

In [ ]:
after_loss, after_accuracy = fine_tuning_model.evaluate(val_ds)

print(f"accuracy avant fine-tuning : {before_accuracy:.3f}")
print(f"accuracy apres fine-tuning : {after_accuracy:.3f}")
print(f"loss avant fine-tuning     : {before_loss:.3f}")
print(f"loss apres fine-tuning     : {after_loss:.3f}")
print(f"evolution accuracy : {(after_accuracy - before_accuracy) * 100:.2f}")
print(f"evolution loss : {after_loss - before_loss:.2f}")

In [ ]:
pip install mlflow 

In [ ]:
import mlflow
import mlflow.tensorflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("NeuroVision")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def train_experiment(model, run_name, model_type, train_ds, val_ds, class_names, learning_rate, batch_size):
    checkpoint_dir = os.path.join("checkpoints", run_name)
    backup_dir = os.path.join("training_backup", run_name)
    artifacts_dir = os.path.join("experiment_artifacts", run_name)

    os.makedirs(checkpoint_dir, exist_ok=True)
    os.makedirs(backup_dir, exist_ok=True)
    os.makedirs(artifacts_dir, exist_ok=True)

    checkpoint_path = os.path.join(checkpoint_dir, "best_model.keras")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    callbacks = [
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1
        ),
        BackupAndRestore(
            backup_dir=backup_dir,
            save_freq="epoch"
        ),
        EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=5,
            restore_best_weights=True,
            verbose=1
        )
    ]

    with mlflow.start_run(run_name=run_name) as run:

        mlflow.log_params({
            "model_type": model_type,
            "learning_rate": learning_rate,
            "epochs_max": epochs,
            "batch_size": batch_size,
            "optimizer": "Adam",
            "loss_function": "categorical_crossentropy",
        })

        start_time = time.time()

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            callbacks=callbacks,
            verbose=1
        )

        duration = time.time() - start_time

        best_model = tf.keras.models.load_model(checkpoint_path)

        val_loss, val_accuracy = best_model.evaluate(val_ds,)

        y_true = []
        y_pred = []

        for images, labels in val_ds:
            predictions = best_model.predict(images)
            labels = labels.numpy()
            labels = np.argmax(labels, axis=1)
            y_true.extend(labels.astype(int).tolist())
            y_pred.extend(np.argmax(predictions, axis=1).tolist())

        val_f1 = f1_score(
            y_true,
            y_pred,
            average="macro",
        )

        mlflow.log_metrics({
            "val_accuracy": float(val_accuracy),
            "val_loss": float(val_loss),
            "val_f1_macro": float(val_f1),
            "duration_seconds": float(duration),
            "epochs_completed": len(history.epoch)
        })

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(
            history.history["accuracy"],
            label="train"
        )
        axes[0].plot(
            history.history["val_accuracy"],
            label="validation"
        )
        axes[0].set_title("accuracy")
        axes[0].set_xlabel("epoque")
        axes[0].legend()
        axes[0].grid(True)

        axes[1].plot(
            history.history["loss"],
            label="train"
        )
        axes[1].plot(
            history.history["val_loss"],
            label="validation"
        )
        axes[1].set_title("loss")
        axes[1].set_xlabel("epoque")
        axes[1].legend()
        axes[1].grid(True)

        fig.tight_layout()

        plot_path = os.path.join(artifacts_dir, "learning_curves.png")
        fig.savefig(plot_path)
        plt.close(fig)

        mlflow.log_artifact(plot_path)

        # 7. Enregistrer le rapport de classification
        report = classification_report(
            y_true,
            y_pred,
            labels=list(range(4)),
            target_names=class_names,
            zero_division=0
        )

        report_path = os.path.join(
            artifacts_dir, "classification_report.txt"
        )

        with open(report_path, "w", encoding="utf-8") as file:
            file.write(report)

        mlflow.log_artifact(report_path)

        mlflow.tensorflow.log_model(best_model, artifact_path="model")

        mlflow.log_artifact(checkpoint_path, artifact_path="checkpoints")

        run_id = run.info.run_id

        print(f"\nexperience : {run_name}")
        print(f"val accuracy : {val_accuracy:.4f}")
        print(f"val loss     : {val_loss:.4f}")
        print(f"val F1 macro : {val_f1:.4f}")
        print(f"duree        : {duration:.2f} secondes")

    return {
        "run_id": run_id,
        "run_name": run_name,
        "val_accuracy": float(val_accuracy),
        "val_loss": float(val_loss),
        "val_f1_macro": float(val_f1),
        "duration_seconds": float(duration)
    }

In [ ]:
class_names = ["glioma", "meningioma", "no_tumor", "pituitary"]

results = []
results.append(train_experiment(
        model=cnn_best_model,
        run_name="cnn_from_scratch",
        model_type="cnn",
        train_ds=train_ds,
        val_ds=val_ds,
        class_names=class_names,
        learning_rate=0.001,
        epochs=15
    )
)

results.append(
    train_experiment(
        model=transfer_lerning_model,
        run_name="MobileNet_transfer_learning",
        model_type="Transfer Learning",
        train_ds=train_ds,
        val_ds=val_ds,
        class_names=class_names,
        epochs=15,
    )
)

results.append(
    train_experiment(
        model=fine_tuning_model,
        run_name="MobileNet_fine_tuning",
        model_type="fine-tuning",
        train_ds=train_ds,
        val_ds=val_ds,
        class_names=class_names,
        epochs=15,
    )
)

In [ ]:
comparison = pd.DataFrame(results)
comparison = comparison[["run_name","val_accuracy","val_loss","val_f1_macro","duration_seconds"]]
comparison.sort_values(by="val_f1_macro", ascending=False)


In [ ]:
from mlflow import MlflowClient
best_result = max(results,key=lambda result: result["val_f1_macro"])

# runs = client.search_runs(
#     experiment_ids=[experiment_id],
#     order_by=["metrics.val_accuracy DESC"],
#     max_results=3
# )


# experiment = client.get_experiment_by_name(experiment_name)

# if experiment is None:
#     experiment_id = client.create_experiment(experiment_name)
# else:
#     experiment_id = experiment.experiment_id

model_name = "NeuroVisionClassifier"
model_uri = f"runs:/{best_result['run_id']}/model"

registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)

client = MlflowClient()

client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=registered_model.version
)

print("Meilleure experience :", best_result["run_name"])
print("Modele enregistre :", model_name)
print("Version champion :", registered_model.version)

In [ ]:
champion_model = mlflow.tensorflow.load_model("models:/NeuroVisionClassifier@champion")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay

y_true = []
y_pred = []

for images, labels in test_ds:
    probabilities = champion_model.predict(images)
    labels = labels.numpy()
    labels = np.argmax(labels, axis=1)
    predictions = np.argmax(probabilities, axis=1)
    y_true.extend(labels.astype(int).tolist())
    y_pred.extend(predictions.astype(int).tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
  
test_accuracy = accuracy_score(y_true, y_pred)

print(f"test accuracy : {test_accuracy:.4f}")
print(f"test accuracy : {test_accuracy * 100:.2f}%\n")

print("rapport de classification :")

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(4)),
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(4))
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(9, 7))

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title("champion model matrice de confusion sur le test")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()